In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [14]:
spark = SparkSession.builder \
    .appName("credit-risk-lakehouse-clientes") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

spark

In [15]:
BASE_PATH = "/app/data"
NUM_CLIENTES = 10_000

df_cliente = spark.range(1, NUM_CLIENTES + 1) \
    .withColumnRenamed("id", "cliente_id") \
    .withColumn("sexo", element_at(array(lit("M"), lit("F")), floor(rand() * 2 + 1).cast("int"))) \
    .withColumn("idade", floor(rand() * 47 + 18).cast("int")) \
    .withColumn("estado_civil", element_at(array(lit("solteiro"), lit("casado"), lit("divorciado"), lit("viuvo"), lit("uniao_estavel")), floor(rand() * 5 + 1).cast("int"))) \
    .withColumn("escolaridade", element_at(array(lit("fundamental"), lit("medio"), lit("superior_incompleto"), lit("superior_completo"), lit("pos_graduacao")), floor(rand() * 5 + 1).cast("int"))) \
    .withColumn("renda_mensal", round(rand() * 18000 + 1200, 2)) \
    .withColumn("tempo_emprego_meses", floor(rand() * 240).cast("int")) \
    .withColumn("uf", element_at(array(lit("PR"), lit("SC"), lit("SP"), lit("RJ"), lit("MG"), lit("RS"), lit("BA"), lit("GO"), lit("PE"), lit("CE")), floor(rand() * 10 + 1).cast("int"))) \
    .withColumn("data_cadastro", date_sub(current_date(), floor(rand() * 1460).cast("int")))

In [16]:
df_cliente.write \
    .mode("overwrite") \
    .parquet(f"{BASE_PATH}/bronze/cliente")

In [18]:
df_cliente_bronze = spark.read.parquet(f"{BASE_PATH}/bronze/cliente")

df_cliente_bronze.show(10, truncate=False)
df_cliente_bronze.count()

+----------+----+-----+-------------+-------------------+------------+-------------------+---+-------------+
|cliente_id|sexo|idade|estado_civil |escolaridade       |renda_mensal|tempo_emprego_meses|uf |data_cadastro|
+----------+----+-----+-------------+-------------------+------------+-------------------+---+-------------+
|1         |F   |53   |divorciado   |pos_graduacao      |3225.78     |99                 |SP |2023-01-02   |
|2         |M   |53   |viuvo        |superior_incompleto|14058.57    |181                |BA |2025-05-04   |
|3         |M   |43   |viuvo        |superior_incompleto|2959.15     |12                 |MG |2025-01-04   |
|4         |F   |34   |viuvo        |pos_graduacao      |18749.14    |167                |MG |2025-12-29   |
|5         |M   |30   |solteiro     |pos_graduacao      |5387.29     |95                 |PE |2023-03-08   |
|6         |M   |60   |uniao_estavel|superior_completo  |18800.43    |76                 |PR |2022-09-07   |
|7         |F   |23

10000

In [20]:
spark.stop()